## bloc 0 -Imports et configuration

In [ ]:
# === Bloc 0 — Imports complets (à mettre tout en haut, une seule fois) ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet

sns.set_theme(style="whitegrid")

## bloc 1 Chargement et diagnostic initial

In [ ]:
import pandas as pd

FILE_PATH = "/kaggle/input/datasets/asmaallaigui/data-stage/Data_stage.xlsx" 
SHEET_NAME = "Sheet1"

df = pd.read_excel("/kaggle/input/datasets/asmaallaigui/data-stage/Data_stage.xlsx", sheet_name= 'Sheet1')

print("Dimensions :", df.shape)
df.head()
df.info()
df.describe()
print(df.isna().sum().sort_values(ascending=False))

## bloc 2 Détection des doublons (granularité métier)

In [ ]:
# Une ligne attendue = agence x office x fournisseur x mois
key_cols = [
    "KEY_F_MD_DATE.Calendar.Year",
    "KEY_F_MD_DATE.Calendar.Month",
    "[Master Account Number]",
    "[Office ID]",
    "[Provider Code]",
]

print("Doublons (toutes colonnes) :", df.duplicated().sum())
print("Doublons (clé métier) :", df.duplicated(subset=key_cols).sum())

## bloc 3 Correction des types (dates, catégories, numériques) 

In [ ]:
# Reconstruction d'une vraie colonne date
df["date"] = pd.to_datetime(
    df["KEY_F_MD_DATE.Calendar.Year"].astype(str)
    + "-"
    + df["KEY_F_MD_DATE.Calendar.Month"].astype(str),
    format="%Y-%b",  # remplace par "%Y-%m" si Month est numérique
    errors="coerce",
)

cat_cols = [
    "[Master Account Number]", "[Master Account Name]",
    "[Office ID]", "[Provider Name]", "[Provider Code]",
]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

measure_cols = [
    "Air Net Bookings", "Air ATLA Confirmed Bookings",
    "Air Net Bookings EAC", "Air Net Bookings NDC",
    "Car Net Bookings", "Hotel Net Bkgs",
    "Insurance Net Bookings", "Rail Net Bookings", "Cruise Bookings",
]
measure_cols = [c for c in measure_cols if c in df.columns]
df[measure_cols] = df[measure_cols].apply(pd.to_numeric, errors="coerce")

## bloc 4 Valeurs manquantes vs zéros structurels 

In [ ]:
print("% de zéros par mesure :")
print((df[measure_cols] == 0).mean().round(3) * 100)

print("\n% de NaN par mesure :")
print(df[measure_cols].isna().mean().round(3) * 100)

# NaN = absence de vente -> remplacés par 0
df[measure_cols] = df[measure_cols].fillna(0)

# Lignes sans date ou sans agence -> exclues
df = df.dropna(subset=["date", "[Master Account Number]"])

## bloc 4 Détection des outliers (méthode IQR)

In [ ]:
def detecter_outliers_iqr(serie: pd.Series) -> pd.Series:
    """Masque booléen des valeurs hors de [Q1-1.5*IQR, Q3+1.5*IQR]."""
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)

outlier_summary = {col: detecter_outliers_iqr(df[col]).sum() for col in measure_cols}
print(pd.Series(outlier_summary).sort_values(ascending=False))

# Flaguées, non supprimées (un gros volume peut être une vraie grosse agence)
df["Air_Net_Bookings_outlier"] = detecter_outliers_iqr(df["Air Net Bookings"])

## bloc 5 Statistiques descriptives (analyse univariée

In [ ]:
print(df[measure_cols].describe().T)
print("\nTop agences :\n", df["[Master Account Name]"].value_counts().head(10))
print("\nTop fournisseurs :\n", df["[Provider Name]"].value_counts().head(10))

## bloc 6 Visualisation : distributions numériques

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
cols_a_tracer = ["Air Net Bookings", "Car Net Bookings", "Hotel Net Bkgs", "Insurance Net Bookings"]
for ax, col in zip(axes.flat, cols_a_tracer):
    sns.histplot(df[col], kde=True, ax=ax, bins=40)
    ax.set_title(f"Distribution : {col}")
plt.tight_layout()
plt.show()

## bloc 7 Visualisation : top fournisseurs (catégoriel) 

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df["[Provider Name]"].value_counts().head(10).plot(kind="bar", ax=ax)
ax.set_title("Top 10 fournisseurs (nombre de lignes)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## bloc 8 Série temporelle globale

In [ ]:
serie_mensuelle = df.groupby("date")["Air Net Bookings"].sum()

fig, ax = plt.subplots(figsize=(12, 5))
serie_mensuelle.plot(ax=ax)
ax.set_title("Evolution mensuelle du total Air Net Bookings")
plt.tight_layout()
plt.show()

## bloc 9 Matrice de corrélation (analyse bivariée)

In [ ]:
corr = df[measure_cols].corr(method="spearman")  # robuste aux distributions asymétriques

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Corrélation entre les types de réservations")
plt.tight_layout()
plt.show()

## bloc 10 Croisement numérique x catégoriel (boxplot)

In [ ]:
top_providers = df["[Provider Name]"].value_counts().head(6).index
subset = df[df["[Provider Name]"].isin(top_providers)]

fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(data=subset, x="[Provider Name]", y="Air Net Bookings", ax=ax)
ax.set_title("Air Net Bookings par fournisseur (top 6)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## bloc 11 Test ANOVA / Kruskal-Wallis (volumes selon fournisseur) 

In [ ]:
groupes = [subset.loc[subset["[Provider Name]"] == p, "Air Net Bookings"] for p in top_providers]

# Test de normalité (sur échantillon, Shapiro est sensible à N élevé)
_, p_normal = stats.shapiro(subset["Air Net Bookings"].sample(min(500, len(subset)), random_state=0))
print(f"Shapiro p-value = {p_normal:.4f}")

if p_normal > 0.05:
    stat, p_val = stats.f_oneway(*groupes)
    print(f"ANOVA : stat={stat:.2f}, p-value={p_val:.4f}")
else:
    stat, p_val = stats.kruskal(*groupes)
    print(f"Kruskal-Wallis : stat={stat:.2f}, p-value={p_val:.4f}")

print("Significatif" if p_val < 0.05 else "Non significatif")

## bloc 12 Test Chi-2 (agence x fournisseur)

In [ ]:
top_agencies = df["[Master Account Name]"].value_counts().head(6).index
subset_cat = df[
    df["[Provider Name]"].isin(top_providers) & df["[Master Account Name]"].isin(top_agencies)
]
table = pd.crosstab(subset_cat["[Master Account Name]"], subset_cat["[Provider Name]"])

chi2, p_chi2, dof, expected = stats.chi2_contingency(table)
print(f"Chi2={chi2:.2f}, p-value={p_chi2:.4f}, ddl={dof}")

## bloc 13 Corrélation entre canaux (NDC vs classique)

In [ ]:
r, p_r = stats.spearmanr(df["Air Net Bookings"], df["Air Net Bookings NDC"])
print(f"Corrélation Air Net Bookings vs NDC : r={r:.3f}, p-value={p_r:.4f}")

In [ ]:
serie = df.groupby("date")["Air Net Bookings"].sum().sort_index()

# S'assurer que l'index est bien une fréquence mensuelle continue (pas de mois manquant)
serie = serie.asfreq("MS")  # MS = Month Start
serie = serie.fillna(0)     # un mois totalement absent = 0 réservation ce mois-là

print(serie.tail(12))
print("\nNombre de points :", len(serie))

In [ ]:
# Agrégation mensuelle brute
serie = df.groupby("date")["Air Net Bookings"].sum().sort_index()
serie = serie.asfreq("MS")

# Identifier les mois avec un volume de lignes anormalement faible (probable donnée incomplète)
lignes_par_mois = df.groupby("date").size()
print("Mois avec le moins de lignes :")
print(lignes_par_mois.sort_values().head(10))

# Seuil arbitraire : un mois avec moins de 50 lignes est suspect (à ajuster selon ce que tu vois ci-dessus)
mois_suspects = lignes_par_mois[lignes_par_mois < 50].index
print("\nMois jugés incomplets/anormaux :", list(mois_suspects))

# On remplace ces mois par NaN (donnée manquante), puis on interpole
serie.loc[serie.index.isin(mois_suspects)] = np.nan
serie = serie.interpolate(method="linear")

print("\nSérie corrigée (12 derniers mois) :")
print(serie.tail(12))

## bloc 14 Décomposition de la série (tendance / saisonnalité / résidu)

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(serie, model="additive", period=12)

fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.show()

In [ ]:
resultat_adf = adfuller(serie.dropna())
print(f"ADF Statistic : {resultat_adf[0]:.4f}")
print(f"p-value : {resultat_adf[1]:.4f}")
print("Stationnaire" if resultat_adf[1] < 0.05 else "Non stationnaire (différenciation nécessaire)")

## bloc 15 Préparation des données (correction anomalie + découpage train/test) 

In [ ]:
# Détection et correction de l'anomalie (mois avec très peu de lignes = collecte incomplète)
lignes_par_mois = df.groupby("date").size()
mois_suspects = lignes_par_mois[lignes_par_mois < 50].index

serie_complete = df.groupby("date")["Air Net Bookings"].sum().sort_index()
serie_complete.loc[serie_complete.index.isin(mois_suspects)] = np.nan
serie_complete = serie_complete.interpolate(method="linear")

# Série restreinte à la période continue et exploitable (2024-2026)
serie_continue = serie_complete.loc["2024-01-01":].asfreq("MS")

n_test = 6
train = serie_continue.iloc[:-n_test]
test = serie_continue.iloc[-n_test:]

print(f"Train : {train.index.min()} à {train.index.max()} ({len(train)} points)")
print(f"Test  : {test.index.min()} à {test.index.max()} ({len(test)} points)")

## bloc 16 Baseline naïve 

In [ ]:
baseline_pred = train.iloc[-n_test:].values


## bloc 17 Holt-Winters (sur série continue 2024-2026)

In [ ]:
modele_hw = ExponentialSmoothing(
    train, trend="add", seasonal="add", seasonal_periods=12,
).fit()

pred_hw = modele_hw.forecast(n_test)
print("Prédictions Holt-Winters :")
print(pred_hw)

## bloc 18 SARIMA (sur série continue 2024-2026)

In [ ]:

modele_sarima = SARIMAX(
    train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False, enforce_invertibility=False,
).fit(disp=False)

pred_sarima = modele_sarima.get_forecast(steps=n_test).predicted_mean
print("Prédictions SARIMA :")
print(pred_sarima)

## bloc 19 Prophet (sur historique COMPLET, 2019 inclus)

In [ ]:

# On exclut de l'entraînement Prophet les 6 derniers mois (même logique de test que les autres modèles)
train_prophet_dates = serie_complete.index[serie_complete.index <= train.index.max()]
train_complet = serie_complete.loc[serie_complete.index.isin(train_prophet_dates) | (serie_complete.index < "2024-01-01")]

df_prophet = train_complet.reset_index()
df_prophet.columns = ["ds", "y"]

modele_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
modele_prophet.fit(df_prophet)

futur = modele_prophet.make_future_dataframe(periods=n_test, freq="MS")
forecast_prophet = modele_prophet.predict(futur)

pred_prophet = forecast_prophet.set_index("ds")["yhat"].loc[test.index]
print("Prédictions Prophet (entraîné avec 2019 inclus) :")
print(pred_prophet)

## bloc 20 Comparaison des 4 approches (inchangé)

In [ ]:
def calculer_metriques(reel, predit, nom):
    mae = mean_absolute_error(reel, predit)
    rmse = np.sqrt(mean_squared_error(reel, predit))
    mape = np.mean(np.abs((reel - predit) / reel)) * 100
    return {"Modèle": nom, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE (%)": round(mape, 2)}

comparaison = pd.DataFrame([
    calculer_metriques(test.values, baseline_pred, "Baseline naïve"),
    calculer_metriques(test.values, pred_hw.values, "Holt-Winters"),
    calculer_metriques(test.values, pred_sarima.values, "SARIMA"),
    calculer_metriques(test.values, pred_prophet.values, "Prophet"),
]).sort_values("MAPE (%)")
print(comparaison)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Ligne pointillée grise = niveau 2019 (référence pré-Covid), affichée séparément
serie_2019 = serie_complete.loc["2019-01-01":"2019-12-01"]
ax.axhline(serie_2019.mean(), color="gray", linestyle=":", label=f"Moyenne 2019 (réf. pré-Covid) : {serie_2019.mean():.0f}")

train.plot(ax=ax, label="Train", color="lightgray")
test.plot(ax=ax, label="Test (réel)", color="black", linewidth=2)
pd.Series(baseline_pred, index=test.index).plot(ax=ax, label="Baseline", linestyle=":")
pred_hw.plot(ax=ax, label="Holt-Winters", linestyle="--")
pred_sarima.plot(ax=ax, label="SARIMA", linestyle="--")
pred_prophet.plot(ax=ax, label="Prophet (avec 2019)", linestyle="--")

ax.legend()
ax.set_title("Comparaison des modèles, avec repère du niveau pré-Covid (2019)")
plt.tight_layout()
plt.show()

## Bloc 21 Validation robuste du modèle gagnant (au-delà d'un seul test) 

In [ ]:
def evaluer_sur_fenetre(nom_modele, date_fin_train, n_test=3, changepoint_prior_scale=0.05):
    """Évalue un modèle sur une fenêtre glissante donnée. nom_modele : 'sarima', 'hw', 'prophet'."""
    train_f = serie_complete.loc[:date_fin_train]
    test_f = serie_complete.loc[date_fin_train:].iloc[1:n_test+1]

    if nom_modele == "sarima":
        m = SARIMAX(train_f, order=(1,1,1), seasonal_order=(1,1,1,12),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        pred = m.get_forecast(steps=n_test).predicted_mean
    elif nom_modele == "hw":
        m = ExponentialSmoothing(train_f, trend="add", seasonal="add", seasonal_periods=12).fit()
        pred = m.forecast(n_test)
    elif nom_modele == "prophet":
        df_p = train_f.reset_index(); df_p.columns = ["ds", "y"]
        m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                    changepoint_prior_scale=changepoint_prior_scale).fit(df_p)
        futur = m.make_future_dataframe(periods=n_test, freq="MS")
        pred = m.predict(futur).set_index("ds")["yhat"].loc[test_f.index]

    return mean_absolute_error(test_f, pred)

dates_coupure = ["2025-10-01", "2026-01-01", "2026-04-01"]

resultats_cv = []
for nom in ["sarima", "hw", "prophet"]:
    maes = [evaluer_sur_fenetre(nom, d) for d in dates_coupure]
    resultats_cv.append({"Modèle": nom, "MAE fenêtre 1": round(maes[0],2),
                          "MAE fenêtre 2": round(maes[1],2), "MAE fenêtre 3": round(maes[2],2),
                          "MAE moyen": round(np.mean(maes), 2)})

print(pd.DataFrame(resultats_cv))

In [ ]:
# Bloc — Identifier et documenter le résidu de démarrage (burn-in)
residus_sarima = modele_sarima.resid
residu_max_idx = residus_sarima.abs().idxmax()

print(f"Résidu le plus extrême : {residus_sarima.loc[residu_max_idx]:.0f}, au mois {residu_max_idx.date()}")
print(f"Position dans la série : point n°{train.index.get_loc(residu_max_idx) + 1} sur {len(train)}")

# Recalcul des statistiques de résidus SANS ce point de démarrage (diagnostic plus juste)
residus_sans_burnin = residus_sarima.drop(residu_max_idx)
print("\nStatistiques des résidus hors point de démarrage :")
print(residus_sans_burnin.describe())

## BLOC 22 Diagnostic des résidus (sur le modèle qui gagne la validation croisée) 

In [ ]:
# À adapter selon le modèle gagnant du Bloc 9 — exemple ici avec Prophet
forecast_in_sample = modele_prophet.predict(df_prophet)
residus = df_prophet["y"].values - forecast_in_sample["yhat"].values

ljung_box = acorr_ljungbox(residus, lags=[12], return_df=True)
print(ljung_box)

fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(residus, lags=20, ax=ax)
plt.tight_layout()
plt.show()

## BLOC 23 Diagnostic des résidus du modèle retenu 

In [ ]:
# Résidus du modèle (disponibles nativement dans l'objet ajusté)
residus_sarima = modele_sarima.resid

print("Nombre de résidus disponibles :", len(residus_sarima))
print(residus_sarima.describe())

# Test de Ljung-Box avec un nombre de lags adapté à la taille réelle des résidus
lags_test = min(5, len(residus_sarima) // 2)
ljung_box_sarima = acorr_ljungbox(residus_sarima, lags=[lags_test], return_df=True)
print(f"\nLjung-Box (lags={lags_test}) :")
print(ljung_box_sarima)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(residus_sarima)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Résidus SARIMA dans le temps")

sns.histplot(residus_sarima, kde=True, ax=axes[1])
axes[1].set_title("Distribution des résidus SARIMA")

plt.tight_layout()
plt.show()

## Bloc 24 Recherche des meilleurs paramètres SARIMA (au lieu du tuning Prophet)

In [ ]:
# Bloc 11 corrigé — recherche AIC valide : d et D restent FIXES, seuls p et q varient
import itertools
import warnings
warnings.filterwarnings("ignore")  # SARIMAX est verbeux sur la convergence, on filtre pour la lisibilité

p = q = range(0, 3)
combinaisons_valides = list(itertools.product(p, q))

resultats_aic = []
for p_val, q_val in combinaisons_valides:
    try:
        m = SARIMAX(
            train,
            order=(p_val, 1, q_val),              # d=1 fixé (confirmé par le test ADF)
            seasonal_order=(1, 1, 1, 12),          # D=1 fixé également
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False)
        resultats_aic.append({"p": p_val, "q": q_val, "AIC": round(m.aic, 2)})
    except Exception:
        continue

aic_df = pd.DataFrame(resultats_aic).sort_values("AIC")
print(aic_df)

In [ ]:
# --- Bloc 12 : à exécuter D'ABORD, avant le bloc Holt-Winters ---
modele_final = SARIMAX(
    serie_continue,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

horizon = 5
forecast_final = modele_final.get_forecast(steps=horizon)

resultat_s2 = forecast_final.predicted_mean.to_frame(name="Prévision")
intervalle = forecast_final.conf_int()
resultat_s2["Borne basse"] = intervalle.iloc[:, 0]
resultat_s2["Borne haute"] = intervalle.iloc[:, 1]

print(resultat_s2)

In [ ]:
# Bloc — Prévision Holt-Winters en parallèle de SARIMA, pour comparer la stabilité
modele_hw_final = ExponentialSmoothing(
    serie_continue, trend="add", seasonal="add", seasonal_periods=12
).fit()

prevision_hw = modele_hw_final.forecast(5)
prevision_hw.index = resultat_s2.index  # aligner sur les mêmes dates (août-décembre 2026)

comparaison_finale = resultat_s2.copy()
comparaison_finale["Prévision Holt-Winters"] = prevision_hw.values
comparaison_finale["Écart SARIMA vs HW"] = comparaison_finale["Prévision"] - comparaison_finale["Prévision Holt-Winters"]

print(comparaison_finale)

## Réentraînement final SARIMA + prévision S2 2026

In [ ]:
# Nouvelle variable : version lissée de la prévision, moins sensible aux oscillations mois-à-mois
comparaison_finale["Prévision SARIMA lissée (moy. 3 mois)"] = (
    comparaison_finale["Prévision"].rolling(window=3, min_periods=1, center=True).mean()
)

print(comparaison_finale[["Prévision", "Prévision SARIMA lissée (moy. 3 mois)"]])

In [ ]:
# Nouvelle variable : mesure la confiance du modèle pour chaque mois prédit
comparaison_finale["Largeur IC"] = comparaison_finale["Borne haute"] - comparaison_finale["Borne basse"]
comparaison_finale["Largeur IC (% de la prévision)"] = (
    comparaison_finale["Largeur IC"] / comparaison_finale["Prévision"] * 100
).round(1)

print(comparaison_finale[["Prévision", "Largeur IC (% de la prévision)"]])

In [ ]:
niveau_2019_moyen = serie_complete.loc["2019-01-01":"2019-12-01"].mean()

comparaison_finale["Écart vs 2019 (%)"] = (
    (comparaison_finale["Prévision"] - niveau_2019_moyen) / niveau_2019_moyen * 100
).round(1)

print(comparaison_finale[["Prévision", "Écart vs 2019 (%)"]])

In [ ]:
# Actuellement, tout est agrégé au niveau global (toutes agences confondues)
# Vérifier s'il existe des agences avec un historique suffisant pour un forecasting individuel
historique_par_agence = df.groupby("[Master Account Name]")["date"].nunique().sort_values(ascending=False)
print("Nombre de mois de données par agence (top 10) :")
print(historique_par_agence.head(10))

print(f"\nNombre d'agences avec au moins 24 mois de données : {(historique_par_agence >= 24).sum()}")

## Visualisation finale 

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

serie_continue.plot(ax=ax, label="Historique (2024-2026)", color="black")

# Prévision brute (ce que montrait déjà le graphique)
comparaison_finale["Prévision"].plot(ax=ax, label="Prévision SARIMA (brute)",
                                       color="orange", linestyle="--", alpha=0.5)

# NOUVEAU : version lissée (Ajout 1), plus lisible
comparaison_finale["Prévision SARIMA lissée (moy. 3 mois)"].plot(
    ax=ax, label="Prévision SARIMA (lissée)", color="darkorange", linewidth=2.5)

ax.fill_between(resultat_s2.index, resultat_s2["Borne basse"], resultat_s2["Borne haute"],
                 color="orange", alpha=0.15, label="Intervalle de confiance")

serie_2019 = serie_complete.loc["2019-01-01":"2019-12-01"]
ax.axhline(serie_2019.mean(), color="gray", linestyle=":", label=f"Moyenne 2019 (pré-Covid) : {serie_2019.mean():.0f}")

ax.legend()
ax.set_title("Prévision Air Net Bookings — S2 2026 (SARIMA brut vs lissé)")
plt.tight_layout()
plt.show()

# PARTIE 2 — Comparaison interannuelle (2019, 2024, 2025, 2026)

In [ ]:
# Reconstruction d'un tableau "mois x année" pour comparaison directe
comparaison_annuelle = serie_complete.reset_index()
comparaison_annuelle.columns = ["date", "Air Net Bookings"]
comparaison_annuelle["Année"] = comparaison_annuelle["date"].dt.year
comparaison_annuelle["Mois"] = comparaison_annuelle["date"].dt.month

# Pivot : une colonne par année, une ligne par mois (1 à 12)
pivot_annuel = comparaison_annuelle.pivot_table(
    index="Mois", columns="Année", values="Air Net Bookings"
)
print(pivot_annuel)

## Bloc B — Superposition des courbes historiques par année

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

annees_a_comparer = [2019, 2024, 2025, 2026]
couleurs = {2019: "gray", 2024: "tab:blue", 2025: "tab:green", 2026: "tab:orange"}

for annee in annees_a_comparer:
    if annee in pivot_annuel.columns:
        style = ":" if annee == 2019 else "-"  # 2019 en pointillé, comme référence
        pivot_annuel[annee].plot(ax=ax, label=str(annee), color=couleurs[annee],
                                   linestyle=style, marker="o", markersize=4)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Fév","Mar","Avr","Mai","Juin","Juil","Août","Sep","Oct","Nov","Déc"])
ax.set_xlabel("Mois")
ax.set_ylabel("Air Net Bookings")
ax.set_title("Comparaison interannuelle du volume de réservations")
ax.legend(title="Année")
plt.tight_layout()
plt.show()

## Bloc C — Ajout de la prévision S2 2026 sur le même graphique 

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for annee in annees_a_comparer:
    if annee in pivot_annuel.columns:
        style = ":" if annee == 2019 else "-"
        pivot_annuel[annee].plot(ax=ax, label=str(annee), color=couleurs[annee],
                                   linestyle=style, marker="o", markersize=4)

# On récupère la valeur de juillet 2026 (dernier point réel) pour créer une jonction visuelle
valeur_juillet = serie_complete.loc["2026-07-01"]

# On construit une série "jonction" : juillet (réel) + prévision lissée (août à décembre)
mois_jonction = [7] + list(comparaison_finale.index.month)
valeurs_jonction = [valeur_juillet] + list(comparaison_finale["Prévision SARIMA lissée (moy. 3 mois)"].values)

ax.plot(mois_jonction, valeurs_jonction,
        color="tab:red", linestyle="--", marker="s", label="Prévision S2 2026 (lissée)")

ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Fév","Mar","Avr","Mai","Juin","Juil","Août","Sep","Oct","Nov","Déc"])
ax.set_xlabel("Mois")
ax.set_ylabel("Air Net Bookings")
ax.set_title("Comparaison interannuelle avec prévision S2 2026 (lissée)")
ax.legend(title="Année")
plt.tight_layout()
plt.show()

## Bloc D — Variation en pourcentage par rapport à 2019 (indicateur de "retour au niveau pré-Covid")

In [ ]:
niveau_2019 = pivot_annuel[2019]

fig, ax = plt.subplots(figsize=(12, 5))
for annee in [2024, 2025, 2026]:
    if annee in pivot_annuel.columns:
        variation_pct = ((pivot_annuel[annee] - niveau_2019) / niveau_2019) * 100
        variation_pct.plot(ax=ax, label=f"{annee} vs 2019", marker="o")

ax.axhline(0, color="black", linestyle="-", linewidth=0.8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Fév","Mar","Avr","Mai","Juin","Juil","Août","Sep","Oct","Nov","Déc"])
ax.set_ylabel("Écart en % vs niveau 2019")
ax.set_title("Écart au niveau pré-Covid, par année et par mois")
ax.legend()
plt.tight_layout()
plt.show()

# PHASE 2 — Préparation des données par agence + variables exogènes 

## bloc 25 Construction du dataset panel (agence × mois) 

In [ ]:
# Dataset "panel" : chaque ligne = une agence, un mois donné
# On agrège TOUTES les mesures disponibles, pas seulement Air Net Bookings
panel = df.groupby(["[Master Account Name]", "date"]).agg({
    "Air Net Bookings": "sum",
    "Air ATLA Confirmed Bookings": "sum",
    "Air Net Bookings NDC": "sum",
    "Car Net Bookings": "sum",
    "Hotel Net Bkgs": "sum",
    "Insurance Net Bookings": "sum",
    "Rail Net Bookings": "sum",
    "Cruise Bookings": "sum",
}).reset_index()

panel.columns = ["agence", "date", "air_bookings", "atla_bookings", "ndc_bookings",
                  "car_bookings", "hotel_bookings", "insurance_bookings",
                  "rail_bookings", "cruise_bookings"]

# On garde uniquement les agences avec un historique suffisant (>=24 mois, comme identifié précédemment)
historique_agence = panel.groupby("agence")["date"].nunique()
agences_valides = historique_agence[historique_agence >= 24].index
panel = panel[panel["agence"].isin(agences_valides)]

print(f"Nombre d'agences retenues : {panel['agence'].nunique()}")
print(f"Nombre de lignes (agence x mois) : {len(panel)}")
print(panel.head())

## Bloc 26 — Variables exogènes temporelles (features calendaires)


In [ ]:
panel["annee"] = panel["date"].dt.year
panel["mois"] = panel["date"].dt.month
panel["trimestre"] = panel["date"].dt.quarter

# Encodage cyclique du mois (capture la saisonnalité mieux qu'un simple numéro 1-12)
panel["mois_sin"] = np.sin(2 * np.pi * panel["mois"] / 12)
panel["mois_cos"] = np.cos(2 * np.pi * panel["mois"] / 12)

# Indicateur saison haute/basse (à ajuster selon ta connaissance métier du secteur voyage)
panel["saison_haute"] = panel["mois"].isin([6, 7, 8, 12]).astype(int)

print(panel[["date", "mois", "mois_sin", "mois_cos", "saison_haute"]].head())

## Bloc 27 — Variables exogènes "lag" et tendance (le cœur du signal prédictif en ML)

In [ ]:
panel = panel.sort_values(["agence", "date"])

# Valeurs passées de la cible — les features les plus importantes en forecasting ML
for lag in [1, 2, 3, 12]:
    panel[f"air_bookings_lag{lag}"] = panel.groupby("agence")["air_bookings"].shift(lag)

# Moyenne mobile récente (tendance de court terme par agence)
panel["air_bookings_ma3"] = panel.groupby("agence")["air_bookings"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

# Poids historique de l'agence (proxy de sa taille/importance relative)
poids_agence = panel.groupby("agence")["air_bookings"].sum()
panel["poids_agence"] = panel["agence"].map(poids_agence)
print(panel[["agence", "date", "air_bookings", "air_bookings_lag1", "air_bookings_lag12", "air_bookings_ma3"]].head(15))



## Bloc 28 — Variable exogène externe (exemple : indicateur macro/saisonnier externe)

In [ ]:
# ============================================================
# Bloc 28 — Construction des variables exogènes
# ============================================================

# --- A. Part du canal NDC par agence et par mois ---
panel = panel.merge(
    df.groupby(["[Master Account Name]", "date"])["Air Net Bookings NDC"].sum().reset_index()
      .rename(columns={"[Master Account Name]": "agence", "Air Net Bookings NDC": "ndc_raw"}),
    on=["agence", "date"], how="left"
)
# Part NDC = NDC / total air (évite division par zéro avec un epsilon)
panel["part_ndc"] = panel["ndc_raw"] / (panel["air_bookings"] + 1e-6)
panel = panel.drop(columns=["ndc_raw"])

# --- B. Indicateur de reprise post-Covid (nombre de mois depuis janvier 2024) ---
panel["mois_depuis_reprise"] = (
    (panel["date"].dt.year - 2024) * 12 + panel["date"].dt.month - 1
)

# --- C. Indicateur Ramadan (dates approximatives 2024-2026, à ajuster si dates précises connues) ---
# Ramadan 2024 : 11 mars - 9 avril | 2025 : 1er mars - 30 mars | 2026 : 18 février - 19 mars
periodes_ramadan = [
    ("2024-03-01", "2024-04-30"),
    ("2025-03-01", "2025-03-31"),
    ("2026-02-01", "2026-03-31"),
]
panel["ramadan"] = 0
for debut, fin in periodes_ramadan:
    masque = (panel["date"] >= debut) & (panel["date"] <= fin)
    panel.loc[masque, "ramadan"] = 1

# --- D. Nombre d'agences actives par mois (marché global, effet de composition) ---
agences_actives_par_mois = df.groupby("date")["[Master Account Name]"].nunique().rename("nb_agences_actives")
panel = panel.merge(agences_actives_par_mois, on="date", how="left")

# --- E. Emplacement réservé pour une vraie variable macro externe (à confirmer avec l'encadrant) ---
# indicateur_externe = pd.read_csv("indice_tourisme.csv", parse_dates=["date"])
# panel = panel.merge(indicateur_externe, on="date", how="left")

print(panel[["agence", "date", "part_ndc", "mois_depuis_reprise", "ramadan", "nb_agences_actives"]].head(10))
print("\nValeurs manquantes après construction :")
print(panel[["part_ndc", "mois_depuis_reprise", "ramadan", "nb_agences_actives"]].isna().sum())

In [ ]:
print([c for c in panel.columns if c in ["part_ndc", "mois_depuis_reprise", "ramadan", "nb_agences_actives"]])

## Bloc 29 — Ajouter les nouvelles variables à la liste features

## Bloc 29 — Préparation finale (encodage + split train/test) 

In [ ]:
from sklearn.preprocessing import LabelEncoder

features = [
    "agence_id", "mois_sin", "mois_cos", "trimestre", "saison_haute",
    "air_bookings_lag1", "air_bookings_lag2", "air_bookings_lag3", "air_bookings_lag12",
    "air_bookings_ma3", "poids_agence",
    "hotel_bookings", "car_bookings", "insurance_bookings",
    "part_ndc", "mois_depuis_reprise", "ramadan", "nb_agences_actives",
]

# 1. Nettoyage des lags manquants
panel_model = panel[panel["agence"] != "-"].dropna(subset=[c for c in panel.columns if "lag" in c])

# 2. Clipper les valeurs négatives à 0 AVANT toute transformation log (corrige le bug)
panel_model["air_bookings"] = panel_model["air_bookings"].clip(lower=0)
for lag_col in [c for c in panel_model.columns if "lag" in c] + ["air_bookings_ma3"]:
    panel_model[lag_col] = panel_model[lag_col].clip(lower=0)

# 3. Créer agence_id
encodeur_agence = LabelEncoder()
panel_model["agence_id"] = encodeur_agence.fit_transform(panel_model["agence"])

# 4. Créer la cible log (maintenant sûre, plus de valeurs négatives)
panel_model["air_bookings_log"] = np.log1p(panel_model["air_bookings"])
cible = "air_bookings_log"

# 5. Nettoyage final — inclut désormais la cible et les features dans le même dropna
panel_model = panel_model.replace([np.inf, -np.inf], np.nan)
panel_model = panel_model.dropna(subset=features + [cible])

# 6. Split temporel
date_limite = panel_model["date"].max() - pd.DateOffset(months=3)
train_panel = panel_model[panel_model["date"] <= date_limite]
test_panel = panel_model[panel_model["date"] > date_limite]

X_train, y_train = train_panel[features], train_panel[cible]
X_test, y_test = test_panel[features], test_panel[cible]

print(f"Train : {X_train.shape}, Test : {X_test.shape}")
print(f"Nombre de features utilisées : {len(features)}")
print(f"Valeurs NaN restantes dans y_train : {y_train.isna().sum()}")
print(f"Valeurs NaN restantes dans y_test  : {y_test.isna().sum()}")

In [ ]:
# Nouveau bloc — cibles par horizon (à insérer juste après le Bloc 29)
panel_model = panel_model.sort_values(["agence", "date"])
for h in range(1, 6):
    panel_model[f"cible_h{h}"] = panel_model.groupby("agence")["air_bookings"].shift(-h)

## Bloc 30 — Entraînement LightGBM

In [ ]:
import lightgbm as lgb

modele_lgb = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42)
modele_lgb.fit(X_train, y_train, categorical_feature=["agence_id"])

pred_lgb_log = modele_lgb.predict(X_test)
pred_lgb = np.expm1(pred_lgb_log)
y_test_reel = np.expm1(y_test)

mae_lgb = mean_absolute_error(y_test_reel, pred_lgb)
print(f"LightGBM — MAE (échelle réelle) : {mae_lgb:.2f}")

## Bloc 31 — Entraînement XGBoost (comparaison)

In [ ]:
import xgboost as xgb

modele_xgb = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
)
modele_xgb.fit(X_train, y_train)

pred_xgb = modele_xgb.predict(X_test)
mae_xgb = mean_absolute_error(y_test, pred_xgb)
print(f"XGBoost — MAE (toutes agences confondues) : {mae_xgb:.2f}")

print(f"\nComparaison : LightGBM={mae_lgb:.0f} vs XGBoost={mae_xgb:.0f}")

## Bloc 32 — Importance des variables (interprétabilité — utile pour Power BI/rapport)

In [ ]:
importance = pd.DataFrame({
    "feature": features,
    "importance": modele_lgb.feature_importances_
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=importance, x="importance", y="feature", ax=ax)
ax.set_title("Importance des variables — LightGBM")
plt.tight_layout()
plt.show()

# PHASE 5 — Validation (cohérence avec le global) 

## Bloc 33 — Vérification de cohérence agence vs global

In [ ]:
# Somme des prédictions par agence sur la période de test, à comparer au total global du modèle SARIMA
somme_predictions_agences = pred_lgb.sum()  # sur la période de test (3 derniers mois, toutes agences)
total_reel_periode = y_test.sum()

print(f"Somme des prédictions par agence (LightGBM) : {somme_predictions_agences:.0f}")
print(f"Total réel sur la même période             : {total_reel_periode:.0f}")
print(f"Écart : {abs(somme_predictions_agences - total_reel_periode) / total_reel_periode * 100:.1f}%")

## Bloc 34 — Réentraînement final + prévision S2 2026 par agence

In [ ]:
# Remplace entièrement l'ancien Bloc 34
predictions_par_horizon = {}

for h in range(1, 6):
    data_h = panel_model.dropna(subset=features + [f"cible_h{h}"])
    X_h = data_h[features]
    y_h = np.log1p(data_h[f"cible_h{h}"].clip(lower=0))  # clip + log, cohérent avec la correction du Problème 2

    modele_h = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42)
    modele_h.fit(X_h, y_h, categorical_feature=["agence_id"])

    derniere_ligne_par_agence = panel_model.sort_values("date").groupby("agence").tail(1)
    pred_h_log = modele_h.predict(derniere_ligne_par_agence[features])
    pred_h = np.expm1(pred_h_log)

    mois_cible = pd.Timestamp("2026-07-01") + pd.DateOffset(months=h)
    predictions_par_horizon[mois_cible] = pd.DataFrame({
        "agence": derniere_ligne_par_agence["agence"].values,
        "date": mois_cible,
        "prevision": pred_h
    })

prevision_par_agence = pd.concat(predictions_par_horizon.values(), ignore_index=True)
print(prevision_par_agence.head(15))

# Vérification : les valeurs de nov/déc ne doivent plus être identiques
print("\nVérification agence par agence (les 5 valeurs doivent être différentes) :")
print(prevision_par_agence[prevision_par_agence["agence"] == prevision_par_agence["agence"].iloc[0]])
data_h3 = panel_model.dropna(subset=features + ["cible_h3"])


pred_h = np.expm1(pred_h_log)
pred_h = np.clip(pred_h, a_min=0, a_max=None)  # <- ligne à ajouter


## Bloc 35 — Vérification finale de cohérence avec le global 

In [ ]:
# Bloc 35 mis à jour — comparer le nouveau total (multi-horizon, clippé) au global SARIMA
total_par_mois = prevision_par_agence.groupby("date")["prevision"].sum()
print("Total prévu (somme des agences, multi-horizon) par mois :")
print(total_par_mois)

print("\nComparaison avec la prévision globale SARIMA :")
print(comparaison_finale[["Prévision"]])

print("\nÉcart relatif par mois :")
ecarts = ((total_par_mois.values - comparaison_finale["Prévision"].values) / comparaison_finale["Prévision"].values * 100)
print(pd.Series(ecarts, index=total_par_mois.index).round(1))

In [ ]:
# Identifier les 10 plus grosses agences (celles qui pèsent vraiment dans le total)
top_agences = panel_model.groupby("agence")["air_bookings"].sum().sort_values(ascending=False).head(10)
print("Top 10 agences par volume historique :")
print(top_agences)

# Regarder leurs prévisions mois par mois, pour voir où ça casse
prevision_top = prevision_par_agence[prevision_par_agence["agence"].isin(top_agences.index)]
pivot_top = prevision_top.pivot(index="agence", columns="date", values="prevision")
print("\nPrévisions des top agences par mois :")
print(pivot_top)

In [ ]:
# Vérifier la distribution de la cible d'entraînement pour l'horizon 3 spécifiquement
for h in range(1, 6):
    data_h = panel_model.dropna(subset=features + [f"cible_h{h}"])
    print(f"Horizon {h} : {len(data_h)} lignes, cible moyenne = {data_h[f'cible_h{h}'].mean():.1f}, "
          f"médiane = {data_h[f'cible_h{h}'].median():.1f}")